# Stage A2 — Canonical Seed-42 Hosted Google Colab Execution Notebook (V1.5)
**Protocol:** Stage A2 V1.5 (Amendment 12)
**Dataset:** HDFS (`SPL-HDFS-001` Canonical Split Authority)
**Scope:** 35,000 Train Sessions (586,577 events) | 7,500 Val Sessions (119,531 events)
**Phase:** Runtime Preparation, Hardware Discovery, CUDA Qualification & Preflight Audit (Zero Real Optimizer Steps)

> **NOTE:** Real empirical training is strictly gated and requires independent authorization after runtime qualification.

In [ ]:
# CELL 1 — Mount Google Drive Durable Storage
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# CELL 2 — Runtime & GPU Discovery
!nvidia-smi
import sys, platform, torch
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device Name:', torch.cuda.get_device_name(0))
    print('CUDA Runtime:', torch.version.cuda)
    print('VRAM (GB):', torch.cuda.get_device_properties(0).total_memory / (1024**3))


In [ ]:
# CELL 3 — Clone Repository into /content/Research & Checkout Approved Commit
%cd /content
!rm -rf /content/Research
!git clone https://github.com/Minhlike/Chuyende.git /content/Research
%cd /content/Research
!git checkout train/ch3-stage-a2-implementation
!git status


In [ ]:
# CELL 4 — Install / Verify Exact Dependencies (Target: PyTorch 2.6.0+cu124)
%cd /content/Research
!pip install -r requirements.txt || true
import torch
print('Verified PyTorch Version:', torch.__version__, '| CUDA:', torch.version.cuda)


In [ ]:
# CELL 5 — Locate Drive HDFS Tarball, Copy to Fast Local Disk, Verify SHA-256
import os, shutil, hashlib
from pathlib import Path

drive_source = Path('/content/drive/MyDrive/Chuyende-stage-a2/datasets/HDFS_1.tar.gz')
if not drive_source.exists():
    drive_source = Path('/content/drive/MyDrive/HDFS_1.tar.gz')

local_dest = Path('/content/stage-a2-data/HDFS_1.tar.gz')
local_dest.parent.mkdir(parents=True, exist_ok=True)

EXPECTED_SHA = '6ca6c5bc2671c66afecee9369a2fdac606bf33997a2494ac66aa411fe3e95169'

if drive_source.exists():
    print('Source on Drive found:', drive_source)
    src_sha = hashlib.sha256(drive_source.read_bytes()).hexdigest()
    print('Source SHA-256:', src_sha)
    assert src_sha == EXPECTED_SHA, f'Source SHA mismatch: {src_sha} != {EXPECTED_SHA}'
    print('Copying to fast local disk:', local_dest)
    shutil.copy2(drive_source, local_dest)

assert local_dest.exists(), f'Local dataset missing at {local_dest}'
dst_sha = hashlib.sha256(local_dest.read_bytes()).hexdigest()
print('Local Copy SHA-256:', dst_sha)
assert dst_sha == EXPECTED_SHA, f'Local copy SHA mismatch: {dst_sha} != {EXPECTED_SHA}'
print('HDFS Data Parity Verified: 100% MATCH.')


In [ ]:
# CELL 6 — Run Colab Bootstrap & Preflight (Dynamic Hardware Discovery)
%cd /content/Research
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
!python scripts/bootstrap_stage_a2_colab.py \
    --repo-dir /content/Research \
    --local-data-dest /content/stage-a2-data/HDFS_1.tar.gz \
    --durable-root /content/drive/MyDrive/Chuyende-stage-a2 \
    --env-lock-output /content/Research/experiments/evidence/stage-a2/preexecution/STAGE-A2-COLAB-EXECUTION-ENVIRONMENT-V1.5.json


In [ ]:
# CELL 7 — Run NON_EMPIRICAL CUDA Deterministic Qualification
%cd /content/Research
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
!python scripts/run_stage_a2_deterministic_qualification.py --device cuda --base-dir /content/Research


In [ ]:
# CELL 8 — Inspect Machine-Collected Environment Lock Candidate
%cd /content/Research
import json
env_p = '/content/Research/experiments/evidence/stage-a2/preexecution/STAGE-A2-COLAB-EXECUTION-ENVIRONMENT-V1.5.json'
with open(env_p) as f:
    env_data = json.load(f)
print(json.dumps(env_data, indent=2))


In [ ]:
# CELL 9 — Run Seed-42 Dry-Run Only (Zero Optimizer Steps)
%cd /content/Research
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
!python scripts/run_stage_a2_five_seed_empirical.py \
    --seed 42 \
    --dry-run \
    --base-dir /content/Research \
    --dataset-path /content/stage-a2-data/HDFS_1.tar.gz \
    --durable-root /content/drive/MyDrive/Chuyende-stage-a2/runs \
    --plan /content/Research/experiments/plans/STAGE-A2-FIVE-SEED-EXECUTION-PLAN-V1.5.json


In [ ]:
# CELL 10 — STOP FOR INDEPENDENT AUTHORIZATION
# DO NOT EXECUTE REAL HDFS TRAINING HERE.
print('=================================================================')
print('STAGE A2 COLAB PREPARATION & QUALIFICATION COMPLETE.')
print('STATUS: PENDING INDEPENDENT LAUNCH AUTHORIZATION FOR SEED 42.')
print('=================================================================')
